In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_08_parquet")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/29 18:06:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_csv = spark.read.csv("/workspace/datasets/08_parquet/pedidos.csv", header= True, inferSchema= True)

In [4]:
df_csv.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [5]:
df_csv.show(5)

+---------+----------+-----------+---------+-----------+-----------+
|pedido_id|cliente_id|data_pedido|   status|      canal|valor_total|
+---------+----------+-----------+---------+-----------+-----------+
|    10001|       452| 2025-10-09|concluido|       site|     473.56|
|    10002|        71| 2025-08-03|concluido|        app|     303.13|
|    10003|       411| 2025-10-06|  enviado|       loja|     281.64|
|    10004|       184| 2025-09-03|concluido|       loja|     281.65|
|    10005|       290| 2025-09-15|concluido|marketplace|     893.22|
+---------+----------+-----------+---------+-----------+-----------+
only showing top 5 rows



In [6]:
df_csv.write.mode("overwrite").parquet("/workspace/datasets/08_parquet/arquivos_parquet")

In [7]:
df_parquet = spark.read.parquet("/workspace/datasets/08_parquet/arquivos_parquet")

In [8]:
df_parquet.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [9]:
df_parquet_select = (
df_parquet.select("cliente_id", "valor_total")
)

In [10]:
df_parquet_select.explain(True)

== Parsed Logical Plan ==
'Project ['cliente_id, 'valor_total]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Project [cliente_id#74, valor_total#78]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Optimized Logical Plan ==
Project [cliente_id#74, valor_total#78]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [cliente_id#74,valor_total#78] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/arquivos_parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,valor_total:double>



In [11]:
df_pushdown = (
    df_parquet.select("cliente_id", "valor_total")
    .filter(F.col("valor_total") > 500)
)

In [12]:
df_pushdown.explain(True)

== Parsed Logical Plan ==
'Filter ('valor_total > 500)
+- Project [cliente_id#74, valor_total#78]
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Filter (valor_total#78 > cast(500 as double))
+- Project [cliente_id#74, valor_total#78]
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Optimized Logical Plan ==
Project [cliente_id#74, valor_total#78]
+- Filter (isnotnull(valor_total#78) AND (valor_total#78 > 500.0))
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Physical Plan ==
*(1) Filter (isnotnull(valor_total#78) AND (valor_total#78 > 500.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [cliente_id#74,valor_total#78] Batched: true, DataFilters: [isnotnull(valor_total#78), (valor_total#78 > 500.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file

In [13]:
df_csv.write.mode("overwrite").parquet("/workspace/datasets/08_parquet/pedidos_normal")

In [14]:
df_parquet_pedidos = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_normal")

In [15]:
df_csv.write.mode("overwrite").partitionBy("status").parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [19]:
df_pedidos_status = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [20]:
df_pedidos_status.explain(True)

== Parsed Logical Plan ==
Relation [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Relation [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] parquet

== Optimized Logical Plan ==
Relation [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<pedido_id:int,cliente_id:int,data_pedido:date,canal:string,valor_total:double>



In [22]:
df_status_pruning = (
    df_pedidos_status.filter(F.col("status") == "concluido")
)

In [23]:
df_status_pruning.explain(True)

== Parsed Logical Plan ==
'Filter ('status = concluido)
+- Relation [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Filter (status#272 = concluido)
+- Relation [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] parquet

== Optimized Logical Plan ==
Filter (isnotnull(status#272) AND (status#272 = concluido))
+- Relation [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#267,cliente_id#268,data_pedido#269,canal#270,valor_total#271,status#272] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [isnotnull(status#272), (status#272 = concluido)], PushedFilters: [], Rea

26/09/29 20:03:27 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 2201402 ms exceeds timeout 120000 ms
26/09/29 20:03:28 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/29 20:03:30 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at 

In [ ]:
spark.stop()